# Part 1 — Baseline DistilBERT Classifier

Self-contained notebook. Re-runs the data load and (where needed) reloads the saved baseline checkpoint so it can execute standalone on Colab.

In [ ]:
!pip install -q transformers torch scikit-learn fairlearn aif360 pandas matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

USE_COLS = ["comment_text", "toxic", "black", "white", "muslim", "jewish"]
IDENTITY_COLS = ["black", "white", "muslim", "jewish"]

raw = pd.read_csv("data/jigsaw-unintended-bias-train.csv", usecols=USE_COLS)
raw["label"] = (raw["toxic"] >= 0.5).astype(int)

sample = raw.sample(n=120_000, random_state=SEED)
train_df, eval_df = train_test_split(
    sample, test_size=20_000, stratify=sample["label"], random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)
print("train:", train_df.shape, "eval:", eval_df.shape)


In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128
CKPT_DIR = "checkpoints/baseline"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ToxicDataset(Dataset):
    def __init__(self, df):
        enc = tokenizer(
            df["comment_text"].astype(str).tolist(),
            truncation=True, padding="max_length", max_length=MAX_LEN,
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {
            "input_ids": torch.tensor(self.input_ids[i]),
            "attention_mask": torch.tensor(self.attention_mask[i]),
            "labels": torch.tensor(self.labels[i]),
        }

train_ds = ToxicDataset(train_df)
eval_ds = ToxicDataset(eval_df)
print("tokenized:", len(train_ds), "train /", len(eval_ds), "eval")

In [ ]:
def compute_metrics(pred):
    logits, labels = pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "auc_roc": roc_auc_score(labels, probs),
    }

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=200,
    seed=SEED,
    report_to="none",
    fp16=torch.backends.mps.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
metrics = trainer.evaluate()
print({k: round(v, 4) for k, v in metrics.items() if k.startswith("eval_")})

trainer.save_model(CKPT_DIR)
tokenizer.save_pretrained(CKPT_DIR)
print(f"saved to {CKPT_DIR}")

In [ ]:
from sklearn.metrics import (confusion_matrix, roc_curve, precision_recall_curve,
                             f1_score)
import matplotlib.pyplot as plt
import seaborn as sns

# Cache eval predictions for plots + threshold sweep
_logits = trainer.predict(eval_ds).predictions
_probs = torch.softmax(torch.tensor(_logits), dim=-1)[:, 1].numpy()
_y = eval_df["label"].values

cm = confusion_matrix(_y, (_probs >= 0.5).astype(int))
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0],
            xticklabels=["pred 0", "pred 1"], yticklabels=["true 0", "true 1"])
axes[0].set_title("Confusion matrix (threshold=0.5)")

fpr, tpr, _ = roc_curve(_y, _probs)
axes[1].plot(fpr, tpr); axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR"); axes[1].set_title("ROC")

prec, rec, _ = precision_recall_curve(_y, _probs)
axes[2].plot(rec, prec); axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")
axes[2].set_title("Precision-Recall")

plt.tight_layout(); plt.show()

In [ ]:
thresh_table = pd.DataFrame({
    "threshold": [0.3, 0.4, 0.5, 0.6, 0.7],
    "f1_macro": [round(f1_score(_y, (_probs >= t).astype(int), average="macro"), 4)
                 for t in [0.3, 0.4, 0.5, 0.6, 0.7]],
})
print(thresh_table.to_string(index=False))

### Choosing the operating threshold

The threshold table above shows macro-F1 at five candidates. I keep **0.5** as the operating threshold for the rest of the assignment unless a higher F1 candidate is clearly better.

**What 0.5 implies about platform priorities.** It's the symmetric choice — penalize false positives and false negatives equally. For content moderation that's a non-trivial position:

- A *lower* threshold (e.g. 0.3) catches more genuinely toxic content (fewer false negatives, less harassment exposure for victims) but **flags more innocent users**. On a platform where over-flagging concentrates on specific demographic groups (which is exactly what Section 3 demonstrates), a low threshold *amplifies* civil-rights harm.
- A *higher* threshold (e.g. 0.7) is the opposite trade: less collateral damage, more genuine toxicity slipping through.

Picking 0.5 is the honest baseline that the rest of the audit operates against. Section 6's pipeline then layers a `[0.4, 0.6]` review band on top, so the model never auto-actions on borderline cases regardless of where the threshold sits.